# MMseqs2

This notebook demonstrates three MMseqs2 tools: `run_mmseqs_clustering` groups sequences by similarity to reduce redundancy, `run_mmseqs_search_proteins` finds homologous proteins in a local FASTA database, and `run_mmseqs_search_genomes` performs nucleotide-against-nucleotide comparisons across genome sequences. MMseqs2 provides BLAST-level sensitivity at orders-of-magnitude faster speeds, making it practical for large-scale analyses.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("mmseqs")
display_overview("mmseqs")
display_docs_section("mmseqs", "Background")

# MMseqs2

> [MMseqs2](https://mmseqs.com/) (Many-against-Many sequence searching) is an ultra-fast tool for searching and clustering huge protein and nucleotide sequence sets. It performs BLAST-like searches 100x faster while maintaining similar sensitivity, making it ideal for large-scale sequence analysis. This module provides interfaces for *Protein Search*, *Genome Search*, and *Sequence Clustering* operations using MMseqs2.

## Background

**What does this tool do?**
MMseqs2 finds regions of similarity between biological sequences by using a two-stage cascaded search algorithm. It first uses k-mer matching to quickly identify candidate sequences, then performs sensitive alignment on the filtered set.

**Why is this important?**
Modern sequencing generates datasets too large for traditional BLAST. MMseqs2 enables:

* Functional annotation: Assign function to millions of predicted genes from metagenomes.
* Database deduplication: Cluster redundant sequences to create non-redundant databases.
* Comparative genomics: Find homologous regions across many genomes simultaneously.
* Protein family classification: Group sequences into evolutionary families at scale.

**Scientific foundation:**
MMseqs2 uses a prefilter-align approach:

1. **[K-mer](https://en.wikipedia.org/wiki/K-mer) Matching**: Extracts short amino acid/nucleotide words and uses a fast k-mer index to find candidate matches.
2. **Ungapped Prefiltering**: Scores candidate matches with ungapped alignment, keeping only promising hits.
3. **Gapped Alignment**: Performs [Smith-Waterman](https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm)-like local alignment on the filtered set.
4. **Clustering**: Uses a greedy [set-cover](https://en.wikipedia.org/wiki/Set_cover_problem) algorithm to group sequences by similarity thresholds.

This cascade dramatically reduces the search space while preserving sensitivity comparable to BLAST.

## Available tools

In [2]:
display_available_tools("mmseqs")

- **`run_mmseqs_clustering()`** — Perform sequence clustering using MMseqs2 to reduce redundancy
- **`run_mmseqs_search_genomes()`** — Execute nucleotide genome-to-genome search workflow
- **`run_mmseqs_search_proteins()`** — Search protein sequences using MMseqs2 with per-sequence results

### `run_mmseqs_clustering`

`run_mmseqs_clustering` groups sequences by sequence identity to eliminate near-redundant members from a dataset. Given a minimum identity threshold, each sequence is assigned to a cluster and one member is designated the representative. This is useful before downstream analyses such as structure prediction or MSA construction, where redundancy inflates compute cost without adding information.

In [3]:
from proto_tools import (
    MmseqsClusteringInput,
    MmseqsClusteringConfig,
    run_mmseqs_clustering,
)

In [4]:
# Display input docs
display_api_reference("mmseqs", "input", "run_mmseqs_clustering")

# Two near-identical GFP sequences and one distinct myoglobin sequence
sequences = [
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK",
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDE",
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG",
]

cluster_input = MmseqsClusteringInput(input_sequences=sequences)

**Input** — `MmseqsClusteringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `input_sequences` | `List[string]` | required | List of sequence strings (protein or nucleotide) for clustering. |
| `sequence_ids` | `array` |  | Optional list of sequence identifiers. If not provided, sequences are assigned sequential IDs (seq\_0, seq\_1, ...). |

In [5]:
# Display config docs
display_api_reference("mmseqs", "config", "run_mmseqs_clustering")

# 90% identity threshold — the two GFP variants should merge into one cluster | see docs above for all fields
cluster_config = MmseqsClusteringConfig(min_seq_id=0.90)

**Config** — `MmseqsClusteringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `min_seq_id` | `number` | `0.6` | Minimum sequence identity threshold (0.0-1.0) for grouping sequences into the same cluster. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run sequence clustering
result_cluster = run_mmseqs_clustering(cluster_input, cluster_config)

Running run_mmseqs_clustering [00:00]

In [7]:
# Display output docs
display_api_reference("mmseqs", "output", "run_mmseqs_clustering")

# GFP variants should cluster together; myoglobin should form its own cluster
print(f"Found {result_cluster.num_clusters} clusters from {len(sequences)} sequences")
for r in result_cluster:
    print(f"  {r.sequence_id}: cluster={r.cluster_id}, representative={r.is_representative}")

**Output** — `MmseqsClusteringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[MmseqsClusterResult]` | required | List of clustering results, one per input sequence. The order matches the input sequences order. |
| `sequence_id` | `string` | required | Identifier of the input sequence. |
| `input_sequence` | `string` | required | The original input sequence. |
| `cluster_id` | `string` | required | Identifier of the cluster (usually the representative's ID). |
| `is_representative` | `boolean` |  | Whether this sequence is the cluster representative. |

Found 2 clusters from 3 sequences
  seq_0: cluster=seq_1, representative=False
  seq_1: cluster=seq_1, representative=True
  seq_2: cluster=seq_2, representative=True


### `run_mmseqs_search_proteins`

`run_mmseqs_search_proteins` finds homologous sequences in a protein FASTA database given one or more query sequences. It runs the full MMseqs2 search pipeline (createdb, createindex, search, convertalis) and returns hits with standard alignment statistics. This is useful for functional annotation, ortholog identification, or validating that a designed protein resembles known structures.

In [8]:
import tempfile
from pathlib import Path

from proto_tools import (
    MmseqsSearchProteinsInput,
    MmseqsSearchProteinsConfig,
    run_mmseqs_search_proteins,
)

In [9]:
# Display input docs
display_api_reference("mmseqs", "input", "run_mmseqs_search_proteins")

# Create a small protein FASTA database with hemoglobin alpha, beta, and myoglobin
tmp_dir = tempfile.mkdtemp(prefix="mmseqs_example_")
db_fasta = Path(tmp_dir) / "proteins.fasta"
db_fasta.write_text(
    ">sp|P69905|HBA_HUMAN Hemoglobin subunit alpha\n"
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH\n"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL\n"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR\n"
    ">sp|P68871|HBB_HUMAN Hemoglobin subunit beta\n"
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST\n"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP\n"
    "ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH\n"
    ">sp|P02144|MYG_HUMAN Myoglobin\n"
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL\n"
    "KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP\n"
    "VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG\n"
)
print(f"Database FASTA written to: {db_fasta}")

# Hemoglobin beta query: expect 100% self-hit and secondary HBA hit at ~43%
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

search_input = MmseqsSearchProteinsInput(
    query_sequences=[query_seq],
    mmseqs_db=str(db_fasta),
)

**Input** — `MmseqsSearchProteinsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `query_sequences` | `List[string]` | required | List of protein sequence strings (amino acid sequences) to search. |
| `sequence_ids` | `array` |  | Optional list of sequence identifiers. If not provided, sequences are assigned sequential IDs (seq\_0, seq\_1, ...). |
| `mmseqs_db` | `string` | required | Path to the target database for searching. Can be: - Path to a FASTA file (MMseqs2 will create a temporary database) - Path to a pre-built MMseqs2 database (created with `mmseqs createdb`) |

Database FASTA written to: /tmp/mmseqs_example_qqtfwv_v/proteins.fasta


In [10]:
# Display config docs
display_api_reference("mmseqs", "config", "run_mmseqs_search_proteins")

# High sensitivity search, return all hits | see docs above for all fields
search_config = MmseqsSearchProteinsConfig(
    sensitivity=7.5,
    threads=2,
    only_top_hits=False,
)

**Config** — `MmseqsSearchProteinsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `split` | `integer` | `0` | Memory management mode (0=auto). |
| `sensitivity` | `number` | `4.0` | Search sensitivity (1.0=fast, 7.5=very sensitive). |
| `only_top_hits` | `boolean` | `True` | If True, keep only the best hit per query sequence. |
| `threads` | `integer` | `96` | Number of CPU threads for parallel processing. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run protein search
result_proteins = run_mmseqs_search_proteins(search_input, search_config)

Running run_mmseqs_search_proteins [00:00]

In [12]:
# Display output docs
display_api_reference("mmseqs", "output", "run_mmseqs_search_proteins")

# HBB self-hit at 100% identity; HBA secondary hit at ~43%
print(f"Found {result_proteins.total_hits} total hits")
for r in result_proteins:
    print(f"  Query {r.query_id}: {r.num_hits} hits")
    for hit in r.hits:
        print(f"    -> {hit.target_id}: {hit.pident:.1f}% identity (evalue={hit.evalue:.2e})")

**Output** — `MmseqsSearchProteinsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[MmseqsSequenceSearchResult]` | required | List of search results, one per input sequence. The order matches the input sequences order. |
| `query_id` | `string` | required | Identifier of the query sequence. |
| `query_sequence` | `string` | required | The input query sequence. |
| `hits` | `List[MmseqsHit]` |  | All hits found for this query, sorted by pident descending. |

Found 2 total hits
  Query seq_0: 2 hits
    -> P68871: 100.0% identity (evalue=4.65e-109)
    -> P69905: 43.4% identity (evalue=1.57e-35)


### `run_mmseqs_search_genomes`

`run_mmseqs_search_genomes` performs nucleotide-against-nucleotide comparisons using the full MMseqs2 workflow. This is useful for comparing coding sequences across species, identifying conserved genomic regions, or aligning genome fragments where protein-level search is not appropriate. Query and target sequences are provided directly as nucleotide strings.

In [13]:
from proto_tools import (
    MmseqsSearchGenomesInput,
    MmseqsSearchGenomesConfig,
    run_mmseqs_search_genomes,
)

In [14]:
# Display input docs
display_api_reference("mmseqs", "input", "run_mmseqs_search_genomes")

# Human hemoglobin alpha and beta coding sequences (partial)
query_genomes = [
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGCAAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCGCTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAGCTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCGACCGTGGACGCCGACCTGGCCAGCGTGAGCACCGTGCTGACCTCCAAATACCGTTAA",
]
target_genomes = [
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGCAAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCGCTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAGCTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCGACCGTGGACGCCGACCTGGCCAGCGTGAGCACCGTGCTGACCTCCAAATACCGTTAA",
    "ATGGTGCACCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAACGTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAGAGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAGGTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGACAACCTCAAGGGCACCTTTGCTACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGATCCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGCAAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAATGCCCTGGCCCACAAGTATCACTAA",
]

genome_input = MmseqsSearchGenomesInput(
    query_genomes=query_genomes,
    target_genomes=target_genomes,
)

**Input** — `MmseqsSearchGenomesInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `query_genomes` | `List[string]` | required | List of nucleotide sequence strings (DNA/RNA) to use as queries. |
| `query_ids` | `array` |  | Optional list of query identifiers. If not provided, sequences are assigned sequential IDs (seq\_0, seq\_1, ...). |
| `target_genomes` | `List[string]` | required | List of nucleotide sequence strings to search against. |
| `target_ids` | `array` |  | Optional list of target identifiers. If not provided, sequences are assigned sequential IDs (target\_0, target\_1, ...). |

In [15]:
# Display config docs
display_api_reference("mmseqs", "config", "run_mmseqs_search_genomes")

# Use 2 threads for the search | see docs above for all fields
genome_config = MmseqsSearchGenomesConfig(threads=2)

**Config** — `MmseqsSearchGenomesConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `search_type` | `integer` | `3` | MMseqs2 search type (3=nucleotide vs nucleotide). |
| `sensitivity` | `number` | `7.5` | Search sensitivity (1.0=fast, 7.5=very sensitive). |
| `threads` | `integer` | `96` | Number of CPU threads for parallel processing. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [16]:
# Run genome search
result_genomes = run_mmseqs_search_genomes(genome_input, genome_config)

Running run_mmseqs_search_genomes [00:00]

In [17]:
# Display output docs
display_api_reference("mmseqs", "output", "run_mmseqs_search_genomes")

# Expect a 100% identity self-hit for the HBA query against itself
print(f"Found {result_genomes.total_hits} total hits")
for r in result_genomes:
    print(f"  Query {r.query_id}: {r.num_hits} hits")
    for hit in r.hits:
        print(f"    -> {hit.target_id}: {hit.pident:.1f}% identity")

**Output** — `MmseqsSearchGenomesOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[MmseqsSequenceSearchResult]` | required | List of search results, one per input query genome. The order matches the input query genomes order. |
| `query_id` | `string` | required | Identifier of the query sequence. |
| `query_sequence` | `string` | required | The input query sequence. |
| `hits` | `List[MmseqsHit]` |  | All hits found for this query, sorted by pident descending. |

Found 1 total hits
  Query seq_0: 1 hits
    -> seq_0: 100.0% identity


## Export Results

Results from any MMseqs2 function can be exported to CSV or JSON for downstream analysis or integration with other tools.

In [18]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result_genomes.export("mmseqs_genome_search", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
